# ArchAI-V1 Kaggle training (free GPU)

Imani runs this notebook. It installs the public `Igadsme/archai` repo, gets NATS-Bench from the official Google Drive source (or a private Kaggle dataset Imani uploaded), trains ArchAI-V1 at a planned ~9 hour GPU scale with periodic checkpoints and resume, evaluates a few generated cells on CIFAR-10, and writes checksummed artifacts plus `metrics.json`.

Read `docs/deploy/KAGGLE_RUNBOOK.md` and `docs/deploy/FREE_TIER_PLAN.md` first. Use **Save & Run All** on a GPU session. Do not start Colab Pro or any paid add-on.

Set `SMOKE = True` only for a minutes-long fixture check (no official archive, no CIFAR-10 download). The real run is `SMOKE = False`.


In [ ]:
# Config — edit before Save & Run All
from pathlib import Path

REPO_URL = "https://github.com/Igadsme/archai.git"
REPO_REF = "feat/complete-archai"  # or a specific commit SHA
SMOKE = False
RESUME = False
SEED = 0
TIME_BUDGET_HOURS = 9.0
CHECKPOINT_MINUTES = 15
OUTPUT = Path("/kaggle/working/archai_v1")
WORKDIR = Path("/kaggle/working/archai-src")

# Path A: official tar is downloaded to TAR (leave ARCHIVE/TABLE as None).
# Path B: set TAR or ARCHIVE to the mounted /kaggle/input/... path, or set TABLE
# to a pre-built cifar10.jsonl. Checksums are always verified for official data.
TAR = Path("/kaggle/working/NATS-tss-v1_0-3ffb9-simple.tar")
ARCHIVE = None  # extracted -simple directory, if you already have one
TABLE = None    # pre-built JSONL, if you already have one

OFFICIAL_TAR_SHA256 = "580fd8f3425fed9f495640b0f12ccb7744f89dec7aff263bd75aed37ab0b8bb6"
OFFICIAL_TREE_SHA256 = "183b6ad84c437a36fbbe03f711cdb3a4bd6f3b8675e1af0536d3975f98fa4ba3"
DRIVE_FILE_ID = "17_saCsj_krKjlCBLOJEpNtzPXArMCqxU"


In [ ]:
# Install the repo (public clone; no GitHub token required)
import subprocess
import sys
from pathlib import Path


def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True, **kwargs)

if not (WORKDIR / ".git").exists():
    run(["git", "clone", "--branch", REPO_REF, "--depth", "1", REPO_URL, str(WORKDIR)])
else:
    run(["git", "-C", str(WORKDIR), "fetch", "--depth", "1", "origin", REPO_REF])
    run(["git", "-C", str(WORKDIR), "checkout", REPO_REF])

run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
# ml = torch + torchvision; benchmark = nats-bench (needed to tabulate the official archive)
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(WORKDIR) + "[ml,benchmark]"])
print("installed", WORKDIR, flush=True)


In [ ]:
# Acquire NATS-Bench (skip entirely when SMOKE or TABLE is set)
import sys
import tarfile
from pathlib import Path

sys.path.insert(0, str(WORKDIR))
from archai_benchmark.integrity import sha256_path


def sha256(path: Path) -> str:
    return sha256_path(path)

if SMOKE:
    print("SMOKE: using the committed fixture; official archive is not required")
elif TABLE and Path(TABLE).is_file():
    print("using pre-built table", TABLE, "sha256", sha256(Path(TABLE)))
else:
    archive = Path(ARCHIVE) if ARCHIVE else None
    tar = Path(TAR) if TAR else None
    if archive is None or not Path(archive).exists():
        if tar is None or not Path(tar).is_file():
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
            import gdown
            dest = Path("/kaggle/working/NATS-tss-v1_0-3ffb9-simple.tar")
            url = f"https://drive.google.com/uc?id={DRIVE_FILE_ID}"
            print("downloading official archive from Google Drive", url, flush=True)
            gdown.download(url, str(dest), quiet=False, fuzzy=True)
            tar = dest
        digest = sha256(Path(tar))
        print("tar sha256", digest, flush=True)
        if digest.lower() != OFFICIAL_TAR_SHA256.lower():
            raise SystemExit(f"tar checksum mismatch: {digest} != {OFFICIAL_TAR_SHA256}")
        extract_root = Path("/kaggle/working")
        print("extracting", tar, flush=True)
        with tarfile.open(tar) as handle:
            handle.extractall(extract_root)
        # Official tarball unpacks a directory named NATS-tss-v1_0-3ffb9-simple
        candidates = list(extract_root.glob("NATS-tss-v1_0-3ffb9-simple"))
        if not candidates:
            raise SystemExit("extracted archive directory not found")
        archive = candidates[0]
    tree = sha256(Path(archive))
    print("tree sha256", tree, "path", archive, flush=True)
    if tree.lower() != OFFICIAL_TREE_SHA256.lower():
        raise SystemExit(f"extracted tree checksum mismatch: {tree} != {OFFICIAL_TREE_SHA256}")
    ARCHIVE = archive
    print("official archive ready at", ARCHIVE, flush=True)


In [ ]:
# Build the CIFAR-10 table if needed (official archive only)
import subprocess
import sys
from pathlib import Path

TABLE_PATH = Path(TABLE) if TABLE else Path("/kaggle/working/table/cifar10.jsonl")
if SMOKE:
    TABLE_PATH = WORKDIR / "datasets/fixtures/natsbench-tss-cifar10-256/cells.jsonl"
    print("SMOKE table", TABLE_PATH)
elif TABLE_PATH.is_file() and TABLE_PATH.stat().st_size > 0:
    print("table exists", TABLE_PATH)
else:
    if ARCHIVE is None:
        raise SystemExit("ARCHIVE is not set; attach a dataset or rerun the acquire cell")
    TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, str(WORKDIR / "scripts/benchmark_table.py"),
           "--benchmark-path", str(ARCHIVE),
           "--benchmark-sha256", OFFICIAL_TREE_SHA256,
           "--output", str(TABLE_PATH)]
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
print("table", TABLE_PATH)


In [ ]:
# Train ArchAI-V1 (planned ~9h GPU; --smoke is a tiny CPU path)
import subprocess
import sys

cmd = [sys.executable, str(WORKDIR / "scripts/kaggle_train_archai_v1.py"),
       "--output", str(OUTPUT),
       "--table", str(TABLE_PATH),
       "--seed", str(SEED),
       "--time-budget-hours", str(TIME_BUDGET_HOURS),
       "--checkpoint-minutes", str(CHECKPOINT_MINUTES)]
if ARCHIVE:
    cmd.extend(["--archive", str(ARCHIVE)])
if SMOKE:
    cmd.append("--smoke")
if RESUME:
    cmd.append("--resume")
print("+", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)
print("artifacts under", OUTPUT)


In [ ]:
# List checksummed outputs for download
import hashlib
from pathlib import Path

metrics = OUTPUT / "metrics.json"
print(metrics.read_text() if metrics.is_file() else "metrics.json missing")
print("--- files ---")
for path in sorted(OUTPUT.glob("*")):
    if path.is_file() and path.suffix in {".pt", ".json"}:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        print(f"{path.name}\t{path.stat().st_size}\t{digest}")
